# Reward encoding in visual cortex: Zhong et al. 2025

This notebook has **two parts**, matching the project proposal.

**Part A — one-session methods pilot (encoding-model ablation).** A single
supervised after-learning session: sample neurons balanced across four
visual-cortical regions, build an expanded design matrix, fit per-neuron ridge
encoding models, and ask whether removing the reward regressor worsens held-out
prediction (with a permutation test on reward timing). This is a careful *methods
check* — it proves the pipeline works and shows what one session looks like.

**Part B — the proposal's actual question, scaled across mice (§8).** *How does
the proportion of reward-encoding neurons — and how strongly they activate —
change during learning, by region?* We apply the **same reward ablation** as
Part A (refit the ridge model without the water-delivery reward block, then
permutation-test the exact reward timing) across many **supervised** mice,
before vs after learning in both training blocks. Supervised only, because the
ablated regressor is water delivery — unsupervised mice receive none, so their
ablation would be zero *by construction*. **Each dot is one mouse**, so the
statistics respect the fact that neurons within a mouse are not independent
replicates.

**Run in Colab:** **Runtime → Run all**. No Drive mount or manual upload needed;
files stream from the official Figshare deposit and are cached. Part A downloads
~258 MB; Part B downloads ~100–180 MB per mouse (cached after the first run).
The Part B defaults (2 mice/stage, 250 neurons/region, 50 permutations) are
sized for a ~20-minute run — adjust `MAX_MICE_PER_CONDITION`,
`N_PER_REGION_OVERTIME`, or `N_PERMUTATIONS_OVERTIME` in the config cell to
trade statistical power for time.

Important interpretation: in **Part A**, neurons are not independent biological
replicates, so the raw `p < 0.05` region percentages are descriptive only. **Part
B** fixes this by making the *mouse* the unit of evidence.

## 1. Setup and configuration

Colab already includes the scientific Python packages used below. The configuration
cell is the only place to change sample size, permutation count, or random seed.


In [ ]:
from __future__ import annotations

import json
import math
import platform
import time
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import sklearn
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.proportion import proportion_confint

sns.set_theme(style="whitegrid", context="notebook")
np.set_printoptions(precision=4, suppress=True)

In [ ]:
SESSION = "VR2_2021_04_06_1"
SEED = 42
N_PER_REGION = 750          # Part A neurons per region (was 250; capped at what a region has)
N_PERMUTATIONS = 199
ALPHAS = np.array([0.01, 0.1, 1.0, 10.0, 100.0, 1000.0])
DATA_DIR = Path("/content/Zhong_et_al_2025")
OUTPUT_DIR = Path("/content")

# Figshare file id, exact filename, and expected byte count (Part A only).
FILES = {
    "behavior": (54183860, "Beh_sup_train1_after_learning.npy", 113_352_887),
    "svd": (54866333, "VR2_2021_04_06_1_SVD_dec.npy", 142_132_066),
    "retinotopy": (54184214, "VR2_2021_04_06_trans.npz", 2_559_726),
}

# ---- Part B: reward ablation across learning, supervised cohort (§8) ----
ARTICLE_ID = 28811129               # the whole Figshare deposit
N_PER_REGION_OVERTIME = 250         # neurons sampled per region per mouse (capped at available)
MAX_MICE_PER_CONDITION = 2          # mice per stage; each mouse = one dot
N_PERMUTATIONS_OVERTIME = 50        # reward-timing permutations per session
PART_B_ALPHA = 1.0                  # fixed ridge alpha (skips per-session grid search for speed)

# The learning-time axis: two training blocks, before vs after learning in each.
STAGE_ORDER = ["train1_before", "train1_after", "train2_before", "train2_after"]

# (cohort, stage, behavior file). Part B ablates the WATER-delivery reward block,
# which only exists in the supervised cohort — unsupervised mice never receive
# water, so an ablation there is zero by construction. Supervised only.
# NOTE: each mouse pulls a ~100-180 MB SVD file (cached after the first run).
# The defaults above are sized for a ~20-minute Run-all; raise them for more power.
OVERTIME_CONDITIONS = [
    ("supervised", "train1_before", "Beh_sup_train1_before_learning.npy"),
    ("supervised", "train1_after",  "Beh_sup_train1_after_learning.npy"),
    ("supervised", "train2_before", "Beh_sup_train2_before_learning.npy"),
    ("supervised", "train2_after",  "Beh_sup_train2_after_learning.npy"),
]

print(f"Python {platform.python_version()} | NumPy {np.__version__} | "
      f"scikit-learn {sklearn.__version__}")
print(f"Seed={SEED}, Part A neurons={4 * N_PER_REGION}, permutations={N_PERMUTATIONS}")
print(f"Part B: reward ablation (refit ΔMSE + permutation, alpha={PART_B_ALPHA:g}), "
      f"up to {N_PER_REGION_OVERTIME}/region, {MAX_MICE_PER_CONDITION} mice/stage, "
      f"{len(STAGE_ORDER)} supervised stages")

## 2. Data download and validation

Files are streamed from the official Figshare deposit and checked against their known
byte counts, so a truncated download fails loudly instead of producing quiet nonsense.
Downloads are cached: rerunning the cell reuses a file that is already the right size.
Zhong's SVD file contains an unused serialized scikit-learn object; this notebook reads
only `U` and `V` from that trusted official file.

In [ ]:
AREA_CODES = {
    "V1": (8,),
    "mHV": (0, 1, 2, 9),
    "lHV": (5, 6),
    "aHV": (3, 4),
}

FRAME_FIELDS = (
    "ft", "ft_trInd", "ft_trInd_odd", "ft_trInd_even", "ft_WallID",
    "ft_Pos", "ft_RunSpeed", "ft_isMoving", "ft_move", "BefCueFr",
    "AftCueFr", "ft_GraySpc", "ft_CorrSpc",
)


@dataclass
class DesignMatrix:
    """The regressors, their names, and which columns belong to which block."""
    raw: np.ndarray  # frames × features, unscaled
    names: list[str]
    blocks: dict[str, np.ndarray]  # block name -> column indices into `raw`
    frame_rate: float


def download_figshare_file(file_id, filename, expected_size, data_dir=DATA_DIR):
    """Download once, then reuse. A wrong byte count means a truncated file."""
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    target = data_dir / filename
    if target.exists() and target.stat().st_size == expected_size:
        print(f"Using cached {filename} ({expected_size / 1e6:.1f} MB)")
        return target

    print(f"Downloading {filename} ({expected_size / 1e6:.1f} MB) …")
    url = f"https://ndownloader.figshare.com/files/{file_id}"
    with requests.get(url, stream=True, timeout=(15, 180)) as response:
        response.raise_for_status()
        with target.open("wb") as handle:
            for chunk in response.iter_content(1024 * 1024):
                handle.write(chunk)

    actual = target.stat().st_size
    if actual != expected_size:
        raise RuntimeError(
            f"Incomplete download for {filename}: expected {expected_size} bytes, "
            f"got {actual}. Rerun this cell to retry."
        )
    return target


def align_behavior_frames(beh, n_frames):
    """Return a copy whose frame-aligned arrays match the number of neural frames."""
    aligned = dict(beh)
    trimmed = []
    for field in FRAME_FIELDS:
        values = np.asarray(beh[field])
        if len(values) == n_frames + 1:
            # The official session has one terminal behavior sample with no SVD frame.
            aligned[field] = values[:n_frames]
            trimmed.append(field)
        elif len(values) == n_frames:
            aligned[field] = values
        else:
            raise ValueError(
                f"{field} has {len(values)} frames but neural data has {n_frames}; "
                "expected an exact match or the documented single terminal sample."
            )
    if trimmed:
        print(f"Dropped the final behavior-only sample from {len(trimmed)} fields.")
    return aligned


def infer_frame_rate(ft):
    """Frame rate from timestamp spacing. Zhong stores MATLAB datenums (days)."""
    ft = np.asarray(ft, dtype=float)
    diffs = np.diff(ft[np.isfinite(ft)])
    dt = float(np.median(diffs[diffs > 0]))
    dt_seconds = dt * 86400.0 if dt < 0.01 else dt
    if not 0.01 <= dt_seconds <= 10.0:
        raise ValueError(f"Implausible neural frame interval: {dt_seconds:.6g} seconds")
    return 1.0 / dt_seconds


def map_regions(iarea):
    """Map Zhong's numeric area codes onto the four regions we compare."""
    iarea = np.asarray(iarea)
    labels = np.full(len(iarea), "unmapped", dtype=object)
    for name, codes in AREA_CODES.items():
        labels[np.isin(iarea, codes)] = name
    return labels.astype(str)


def balanced_sample(regions, n_per_region, seed=42):
    """Pick up to n_per_region neurons per region, blind to activity and model fit.

    If a region has fewer neurons than requested we take all of them (and say so)
    instead of crashing, so the same call works on tiny and huge sessions alike.
    Regions can therefore end up with different counts — that is fine, because we
    always report *percentages* with a per-region denominator downstream.
    """
    regions = np.asarray(regions)
    rng = np.random.default_rng(seed)
    selected, labels = [], []
    for region in AREA_CODES:
        candidates = np.flatnonzero(regions == region)
        n = min(n_per_region, len(candidates))
        if n == 0:
            print(f"  {region}: no mapped neurons, skipping")
            continue
        if n < n_per_region:
            print(f"  {region}: only {n} neurons available (< {n_per_region} requested)")
        selected.append(np.sort(rng.choice(candidates, n, replace=False)))
        labels.extend([region] * n)
    return np.concatenate(selected), np.asarray(labels)


def reconstruct_selected(svd, selected):
    """Rebuild activity for just the sampled neurons from the 400-component SVD."""
    U = np.asarray(svd["U"], dtype=np.float32)  # components × neurons
    V = np.asarray(svd["V"], dtype=np.float32)  # components × frames
    return (U[:, np.asarray(selected, dtype=int)].T @ V).T.astype(np.float32, copy=False)

In [ ]:
def _basis_centers(low, high, n_basis):
    """Evenly spaced bump centers, each bump overlapping its neighbours."""
    if not np.isfinite(low) or not np.isfinite(high) or high <= low or n_basis <= 0:
        raise ValueError(f"Invalid raised-cosine range: {low} to {high}, {n_basis} bases")
    centers = np.linspace(low, high, n_basis)
    width = max((high - low) / max(n_basis - 1, 1) * 1.5, np.finfo(float).eps)
    return centers, width


def _cosine_bump(distance):
    """1 at a basis centre, tapering smoothly to 0 at |distance| = 1."""
    values = np.zeros(distance.shape, dtype=np.float32)
    inside = np.isfinite(distance) & (np.abs(distance) <= 1)
    values[inside] = (0.5 + 0.5 * np.cos(np.pi * distance[inside])).astype(np.float32)
    return values


def raised_cosine_basis(values, low, high, n_basis):
    """Soft-bin a continuous variable (position, speed) into overlapping bumps."""
    centers, width = _basis_centers(low, high, n_basis)
    values = np.asarray(values, dtype=float)
    return _cosine_bump((values[:, None] - centers[None, :]) / width)


def event_basis(n_frames, events, frame_rate, start_s, end_s, n_basis):
    """Spread each event over lag bumps, so a neuron can respond before or after it."""
    if n_frames <= 0 or frame_rate <= 0:
        raise ValueError("Invalid event-basis dimensions")
    centers, width = _basis_centers(start_s, end_s, n_basis)
    output = np.zeros((int(n_frames), int(n_basis)), dtype=np.float32)
    events = np.asarray(events, dtype=float).ravel()
    for event in events[np.isfinite(events)]:
        first = max(0, int(math.floor(event + start_s * frame_rate)))
        last = min(n_frames - 1, int(math.ceil(event + end_s * frame_rate)))
        if last < first:
            continue  # the event's whole window falls outside the recording
        frames = np.arange(first, last + 1)
        lags = (frames - event) / frame_rate
        output[frames] += _cosine_bump((lags[:, None] - centers[None, :]) / width)
    return output


def odd_even_masks(frame_trials):
    """Use first/third/fifth/… trials to train, matching Zhong's 'odd' mask."""
    frame_trials = np.asarray(frame_trials, dtype=float)
    valid = np.isfinite(frame_trials)
    trial_ids = np.unique(np.rint(frame_trials[valid]).astype(np.int64))
    if len(trial_ids) < 2:
        raise ValueError("At least two trials are required")
    train = valid & np.isin(np.rint(frame_trials), trial_ids[::2])
    test = valid & np.isin(np.rint(frame_trials), trial_ids[1::2])
    if not train.any() or not test.any():
        raise ValueError("Odd/even trial split is empty")
    return train, test


def _append_block(arrays, names, blocks, block_name, values, column_names):
    """Add one group of regressors and remember which columns it occupies."""
    values = np.asarray(values, dtype=np.float32)
    if values.ndim == 1:
        values = values[:, None]
    start = sum(array.shape[1] for array in arrays)
    arrays.append(values)
    names.extend(column_names)
    blocks[block_name] = np.arange(start, start + values.shape[1], dtype=int)


def build_design_matrix(beh, train_mask, reward_events=None):
    """Build every regressor described in design-matrix-flowchart.pdf.

    Only the speed range is estimated from data, and only from training frames,
    so the held-out trials stay untouched.
    """
    required = [
        "ft", "ft_WallID", "ft_Pos", "ft_RunSpeed", "ft_isMoving",
        "ft_trInd", "LickFr", "RewardFr", "BefCueFr", "AftCueFr",
        "ft_GraySpc", "ft_CorrSpc", "StartFr", "GrayFr", "EndFr",
    ]
    missing = [field for field in required if field not in beh]
    if missing:
        raise KeyError(f"Missing behavior fields: {missing}")
    n_frames = len(np.asarray(beh["ft"]))
    train_mask = np.asarray(train_mask, dtype=bool)
    frame_rate = infer_frame_rate(beh["ft"])
    arrays, names, blocks = [], [], {}

    # Which corridor is on screen.
    wall = np.asarray(beh["ft_WallID"]).astype(str)
    stimuli = sorted(value for value in np.unique(wall) if value.lower() != "nan")
    if len(stimuli) < 2:
        raise ValueError(f"Expected at least two stimuli, found {stimuli}")
    stimulus = np.column_stack([wall == value for value in stimuli]).astype(np.float32)
    _append_block(arrays, names, blocks, "stimulus", stimulus,
                  [f"stimulus:{value}" for value in stimuli])

    # Where the mouse is, and where it is within each corridor.
    position = np.asarray(beh["ft_Pos"], dtype=float)
    position_basis = raised_cosine_basis(position, 0.0, 60.0, 12)
    _append_block(arrays, names, blocks, "position", position_basis,
                  [f"position_basis:{i}" for i in range(12)])
    interactions = (stimulus[:, :, None] * position_basis[:, None, :]).reshape(n_frames, -1)
    _append_block(
        arrays, names, blocks, "position_x_stimulus", interactions,
        [f"position_x_stimulus:{stim}:{i}" for stim in stimuli for i in range(12)],
    )

    # How fast it is running.
    speed = np.asarray(beh["ft_RunSpeed"], dtype=float)
    train_speed = speed[train_mask & np.isfinite(speed)]
    speed = np.where(np.isfinite(speed), speed, np.median(train_speed))
    low, high = np.percentile(train_speed, [1, 99])
    speed_basis = raised_cosine_basis(speed, float(low), float(max(high, low + 1.0)), 5)
    movement_values = np.column_stack([
        speed,
        speed_basis,
        np.asarray(beh["ft_isMoving"], dtype=np.float32),
        np.diff(speed, prepend=speed[0]),
    ])
    _append_block(
        arrays, names, blocks, "movement", movement_values,
        ["running_speed"] + [f"speed_basis:{i}" for i in range(5)]
        + ["is_moving", "acceleration"],
    )

    # Reward: the block this whole notebook is about. -3 to +3 s covers
    # anticipation before delivery and the response after it.
    if reward_events is None:
        reward_events = np.asarray(beh["RewardFr"], dtype=float)
    reward = event_basis(n_frames, reward_events, frame_rate, -3.0, 3.0, 8)
    _append_block(arrays, names, blocks, "reward", reward,
                  [f"reward_lag_basis:{i}" for i in range(8)])

    # Licking and the sound cue: the nuisance signals most confused with reward.
    lick = event_basis(n_frames, beh["LickFr"], frame_rate, -1.0, 2.0, 8)
    _append_block(arrays, names, blocks, "lick", lick,
                  [f"lick_lag_basis:{i}" for i in range(8)])

    cue_field = "SoundDelayFr" if "SoundDelayFr" in beh else "SoundFr"
    cue = event_basis(n_frames, beh[cue_field], frame_rate, 0.0, 3.0, 8)
    _append_block(arrays, names, blocks, "cue", cue,
                  [f"cue_lag_basis:{i}" for i in range(8)])

    # Trial structure: which phase of the trial, and the corridor landmarks.
    epoch = np.column_stack([
        beh["BefCueFr"], beh["AftCueFr"], beh["ft_GraySpc"], beh["ft_CorrSpc"],
    ]).astype(np.float32)
    _append_block(arrays, names, blocks, "epoch", epoch,
                  ["before_cue", "after_cue", "gray_space", "corridor_space"])

    for source, label in [("StartFr", "start"), ("GrayFr", "gray"), ("EndFr", "end")]:
        landmark = event_basis(n_frames, beh[source], frame_rate, 0.0, 2.0, 6)
        _append_block(arrays, names, blocks, f"landmark_{label}", landmark,
                      [f"landmark_{label}_basis:{i}" for i in range(6)])

    raw = np.column_stack(arrays).astype(np.float32, copy=False)
    if not np.isfinite(raw).all():
        bad = np.flatnonzero(~np.isfinite(raw).all(axis=0))
        raise ValueError(f"Design matrix has non-finite columns: {bad[:10].tolist()}")
    return DesignMatrix(raw=raw, names=names, blocks=blocks, frame_rate=frame_rate)

### Aside — what the "lag basis" columns actually do

A reward (or lick, or cue) happens at a single instant, but a neuron's response
is **smeared across time**: some neurons ramp up a second or two *before* the
event (anticipation), others fire *after* it, each with its own delay and
duration. If we handed the model a single "reward happened here" column, it could
only learn *one* number per neuron — an average response pinned to the exact
frame of delivery — and it would miss anticipation entirely.

The lag bases fix that. `event_basis` takes each event and spreads it over a row
of overlapping, bell-shaped **time bumps** tiling a window (reward: −3 s to +3 s,
8 bumps). The picture to hold in your head is a **graphic equaliser, but for
*time-since-event* instead of audio pitch**: instead of one volume knob for
"reward", the neuron gets ~8 sliders — "activity 3 s before", "2 s before", …,
"3 s after" — and ridge regression sets each slider independently.

- A pure **anticipatory** neuron turns up the *before* sliders.
- A **reward-consumption** neuron turns up the *after* sliders.
- The fitted slider values *are* that neuron's reward time-course — which is
  exactly what you read off the coefficient plots in §7.

Two flavours appear in the design matrix, for the same reason (let the model bend
smoothly instead of committing to one hard value):

- **`event_basis`** (reward, lick, cue, landmarks): bumps in **time** around
  discrete events — the lag window above.
- **`raised_cosine_basis`** (position, speed): bumps along a **continuous axis** —
  softly binning "where in the corridor" or "how fast" into overlapping ranges.

*Why overlapping?* Because neighbouring bumps overlap, any real event lands partly
on two or three sliders at once, so the fitted response comes out as a smooth
curve rather than a blocky staircase. That is the whole trick: a handful of
overlapping bumps buys you a flexible, smooth response shape with only a few
parameters to fit.

## 3. Balanced neuron sampling

Region membership comes from the retinotopy file. We sample exactly 250 neurons from
each mapped region with a fixed seed, independently of activity or model fit, then
reconstruct only those neurons from Zhong's 400-component SVD representation.


## 4. Expanded design matrix

The matrix follows `design-matrix-flowchart.pdf`: stimulus identity; 12 position
bases; position×stimulus interactions; speed, speed bases, movement and acceleration;
reward, lick and cue lag bases; cue/space epochs; and corridor landmarks. Reward bases
span −3 to +3 seconds to cover anticipation and delivery.

Columns are standardized with a `StandardScaler` **fitted on training frames only**, so
nothing about the held-out trials leaks into the fit. `Ridge` fits its own unpenalized
intercept. A few basis columns may be constant (for example, position bases covering
stretches the mouse never reaches); the scaler maps those to zero and ridge gives them
zero weight, so they cost nothing and are left in place.

In [ ]:
def regression_metrics(y_true, y_pred):
    """Held-out error and variance explained, one value per neuron."""
    mse = mean_squared_error(y_true, y_pred, multioutput="raw_values")
    r2 = r2_score(y_true, y_pred, multioutput="raw_values")
    return mse, r2


def fit_reward_models(X, Y, train_mask, test_mask, reward_indices, alpha):
    """Fit the full model, then refit without the reward block.

    delta_mse_refit = MSE(reduced) - MSE(full). Positive means reward timing
    bought held-out accuracy that no other regressor could supply. Negative is a
    real answer too: reward added nothing for that neuron.
    """
    reduced_indices = np.setdiff1d(np.arange(X.shape[1]), reward_indices)

    full = Ridge(alpha=alpha).fit(X[train_mask], Y[train_mask])
    mse_full, r2_full = regression_metrics(Y[test_mask], full.predict(X[test_mask]))

    reduced = Ridge(alpha=alpha).fit(X[train_mask][:, reduced_indices], Y[train_mask])
    mse_reduced, r2_reduced = regression_metrics(
        Y[test_mask], reduced.predict(X[test_mask][:, reduced_indices])
    )
    return {
        "full_model": full,
        "reduced_model": reduced,
        "mse_full": mse_full,
        "r2_full": r2_full,
        "mse_reduced": mse_reduced,
        "r2_reduced": r2_reduced,
        "delta_mse_refit": mse_reduced - mse_full,
        "delta_r2_refit": r2_full - r2_reduced,
    }


def permutation_pvalues(observed, null):
    """Fraction of shuffles that matched or beat the real effect (+1 smoothing)."""
    observed = np.asarray(observed, dtype=float)[None, :]
    null = np.asarray(null, dtype=float)
    return (1 + np.sum(null >= observed, axis=0)) / (null.shape[0] + 1)

## 5. Ridge encoding model

The first, third, fifth, … observed trials train the model; the alternating trials are
held out. This matches Zhong's supplied `ft_trInd_odd`/`ft_trInd_even` convention even
though trial IDs are zero-based, and the cell below asserts that our derived masks equal
theirs.

`GridSearchCV` picks one shared ridge alpha using three `GroupKFold` folds *inside* the
training trials, grouped by trial so no trial is split across a fold boundary. The
held-out even trials are never used for tuning — they are only ever scored once, at the
end.

In [ ]:
def permute_reward_events(reward_events, trial_starts, trial_ends, rng):
    """Shuffle each trial's reward offset onto a different trial.

    Reward still lands on a rewarded trial, at a plausible within-trial offset —
    only the exact timing is broken. That is the thing we want to test.
    """
    reward_events = np.asarray(reward_events, dtype=float)
    starts = np.asarray(trial_starts, dtype=float)
    ends = np.asarray(trial_ends, dtype=float)
    valid = np.isfinite(reward_events) & np.isfinite(starts) & np.isfinite(ends)
    if valid.sum() < 2:
        raise ValueError("At least two rewarded trials are required for permutation")
    offsets = rng.permutation(reward_events[valid] - starts[valid])
    return np.clip(starts[valid] + offsets, starts[valid], ends[valid])


def reward_permutation_null(
    design, scaler, Y, train_mask, test_mask, beh, alpha, mse_reduced,
    n_permutations=199, seed=42, progress=True,
):
    """Null distribution for delta_mse_refit under shuffled reward timing.

    Each permutation rebuilds only the reward columns, rescales with the training
    scaler, and refits. Everything else about the model is held fixed.
    """
    reward_indices = design.blocks["reward"]
    rng = np.random.default_rng(seed)
    null = np.empty((n_permutations, Y.shape[1]), dtype=np.float32)
    started = time.perf_counter()
    for permutation in range(n_permutations):
        events = permute_reward_events(
            beh["RewardFr"], beh["StartFr"], beh["EndFr"], rng
        )
        raw = design.raw.copy()
        raw[:, reward_indices] = event_basis(
            len(raw), events, design.frame_rate, -3.0, 3.0, len(reward_indices)
        )
        X = scaler.transform(raw)
        model = Ridge(alpha=alpha).fit(X[train_mask], Y[train_mask])
        mse_permuted = mean_squared_error(
            Y[test_mask], model.predict(X[test_mask]), multioutput="raw_values"
        )
        null[permutation] = (mse_reduced - mse_permuted).astype(np.float32)
        if progress and ((permutation + 1) % 20 == 0 or permutation == 0):
            elapsed = time.perf_counter() - started
            print(f"Permutation {permutation + 1}/{n_permutations} | {elapsed:.1f} s")
    return null


def run_synthetic_recovery_test():
    """Inject a known reward response into one fake neuron; check the pipeline finds it.

    Neuron 0 is driven by a reward basis; the other eight are driven by nuisance
    only. If this fails, the pipeline is broken and the real results mean nothing.
    """
    rng = np.random.default_rng(7)
    n_trials, frames_per_trial = 40, 15
    n_frames = n_trials * frames_per_trial
    trials = np.repeat(np.arange(n_trials), frames_per_trial)
    train, test = odd_even_masks(trials.astype(float))
    starts = np.arange(n_trials) * frames_per_trial
    ends = starts + frames_per_trial - 1
    events = starts + np.where(np.arange(n_trials) % 3 == 0, 4, 10)

    reward = event_basis(n_frames, events, 5.0, -1.0, 1.0, 4)
    nuisance = rng.normal(size=(n_frames, 3))
    design = DesignMatrix(
        raw=np.column_stack([nuisance, reward]).astype(np.float32),
        names=[f"x{i}" for i in range(7)],
        blocks={"nuisance": np.arange(3), "reward": np.arange(3, 7)},
        frame_rate=5.0,
    )
    scaler = StandardScaler().fit(design.raw[train])
    X = scaler.transform(design.raw)

    y_injected = 4.0 * reward[:, 2] + 0.4 * nuisance[:, 0] + rng.normal(0, 0.15, n_frames)
    null_targets = nuisance @ rng.normal(size=(3, 8)) + rng.normal(0, 0.8, (n_frames, 8))
    Y = np.column_stack([y_injected, null_targets]).astype(np.float32)

    results = fit_reward_models(X, Y, train, test, design.blocks["reward"], alpha=0.1)
    null = reward_permutation_null(
        design, scaler, Y, train, test,
        {"RewardFr": events, "StartFr": starts, "EndFr": ends},
        alpha=0.1, mse_reduced=results["mse_reduced"],
        n_permutations=39, seed=9, progress=False,
    )
    p = permutation_pvalues(results["delta_mse_refit"], null)
    output = {
        "injected_delta_mse": float(results["delta_mse_refit"][0]),
        "injected_p": float(p[0]),
        "median_null_p": float(np.median(p[1:])),
    }
    if not (output["injected_delta_mse"] > 0 and
            output["injected_p"] < output["median_null_p"]):
        raise AssertionError(f"Synthetic recovery failed: {output}")
    return output

## 6. Reward ablation and permutation test

The question is whether reward timing tells us anything about a neuron that the other
regressors cannot already supply. We answer it by refitting:

- Fit the **full** model, then drop the entire reward block and **refit** the remaining
  weights. `ΔMSE = MSE_reduced − MSE_full` is reward's unique held-out value.
- The refit matters. If we merely blanked the reward input while keeping the fitted
  weights, the other regressors would never get their chance to compensate, and reward
  would be over-credited.

For inference, reward timing offsets are shuffled among rewarded trials. This keeps
reward on the rewarded stimulus while breaking exact timing, so the null asks
specifically about *timing* rather than about *which corridor was rewarded*. All 1,000
neurons receive all permutations; nothing pre-selects which neurons are tested.

In [ ]:
synthetic_check = run_synthetic_recovery_test()
print("Synthetic recovery passed:", synthetic_check)


In [ ]:
run_started = time.perf_counter()
paths = {
    key: download_figshare_file(file_id, filename, size)
    for key, (file_id, filename, size) in FILES.items()
}

behavior_sessions = np.load(paths["behavior"], allow_pickle=True).item()
if SESSION not in behavior_sessions:
    raise KeyError(f"Session {SESSION!r} not found; available: {list(behavior_sessions)}")
beh_original = behavior_sessions[SESSION]

# The official file includes an unused sklearn model; U and V are plain arrays.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Trying to unpickle estimator TruncatedSVD")
    svd_file = np.load(paths["svd"], allow_pickle=True).item()
svd = {"U": np.asarray(svd_file["U"]), "V": np.asarray(svd_file["V"])}
retin = np.load(paths["retinotopy"], allow_pickle=False)

n_frames = svd["V"].shape[1]
beh = align_behavior_frames(beh_original, n_frames)
frame_rate = infer_frame_rate(beh["ft"])
regions_all = map_regions(retin["iarea"])

qc = pd.DataFrame({
    "item": ["neural frames", "behavior trials", "reward events", "lick events",
             "cue events", "frame rate (Hz)", "SVD neurons"],
    "value": [n_frames, int(beh["ntrials"]), int(np.isfinite(beh["RewardFr"]).sum()),
              len(beh["LickFr"]), int(np.isfinite(beh["SoundFr"]).sum()),
              round(frame_rate, 3), svd["U"].shape[1]],
})
display(qc)


In [ ]:
selected_indices, selected_regions = balanced_sample(
    regions_all, N_PER_REGION, seed=SEED
)
Y = reconstruct_selected(svd, selected_indices)
if Y.shape != (n_frames, len(selected_indices)):
    raise ValueError(f"Unexpected reconstructed target shape: {Y.shape}")
if not np.isfinite(Y).all():
    raise ValueError("Reconstructed neural targets contain non-finite values")

region_counts = pd.Series(selected_regions).value_counts().reindex(AREA_CODES)
display(region_counts.rename("selected neurons").to_frame())
print(f"Target Y shape: {Y.shape} (frames × neurons), dtype={Y.dtype}")

In [ ]:
train_mask, test_mask = odd_even_masks(beh["ft_trInd"])
supplied_train = np.asarray(beh["ft_trInd_odd"], dtype=bool)
supplied_test = np.asarray(beh["ft_trInd_even"], dtype=bool)
if not (np.array_equal(train_mask, supplied_train) and
        np.array_equal(test_mask, supplied_test)):
    raise ValueError("Derived alternating-trial masks do not match Zhong's supplied masks")

design = build_design_matrix(beh, train_mask)
scaler = StandardScaler().fit(design.raw[train_mask])  # training frames only
X = scaler.transform(design.raw)
reward_indices = design.blocks["reward"]

# train_mask already excludes frames with no trial, so the -1 filler never reaches groups.
trial_ids = np.nan_to_num(np.asarray(beh["ft_trInd"], dtype=float), nan=-1).astype(np.int64)
train_groups = trial_ids[train_mask]

manifest = pd.DataFrame(
    [{"block": block, "columns": len(indices)} for block, indices in design.blocks.items()]
)
display(manifest)
n_constant = int((design.raw[train_mask].std(axis=0) == 0).sum())
print(f"Design matrix: {X.shape} (frames × features); "
      f"{n_constant} column(s) constant in training, which ridge gives zero weight")
print(f"Train frames: {train_mask.sum():,}; held-out frames: {test_mask.sum():,}")

# Compact diagnostic: representative varying columns only, so labels stay readable.
varying = np.flatnonzero(design.raw[train_mask].std(axis=0) > 0)
picks = varying[np.unique(np.linspace(0, len(varying) - 1, min(30, len(varying))).astype(int))]
corr = np.corrcoef(X[train_mask][:, picks], rowvar=False)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, vmin=-1, vmax=1, cmap="vlag", center=0, ax=ax,
            xticklabels=[design.names[i] for i in picks],
            yticklabels=[design.names[i] for i in picks])
ax.set_title("Representative training-feature correlations")
plt.tight_layout()
plt.show()

In [ ]:
print("Selecting ridge alpha using grouped folds inside training trials …")
alpha_started = time.perf_counter()

# GroupKFold keeps whole trials together, so no trial straddles a fold boundary.
# refit=False: we only want the alpha here; fit_reward_models does the real fits.
search = GridSearchCV(
    Ridge(),
    {"alpha": ALPHAS},
    cv=GroupKFold(n_splits=3),
    scoring="neg_mean_squared_error",
    refit=False,
)
search.fit(X[train_mask], Y[train_mask], groups=train_groups)
best_alpha = float(search.best_params_["alpha"])
alphas_tried = np.array(search.cv_results_["param_alpha"], dtype=float)
val_mse = -search.cv_results_["mean_test_score"]
print(f"Selected alpha={best_alpha:g} in {time.perf_counter() - alpha_started:.1f} s")

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(alphas_tried, val_mse, marker="o")
ax.axvline(best_alpha, color="tab:red", linestyle="--", label=f"selected {best_alpha:g}")
ax.set(xlabel="ridge alpha", ylabel="mean validation MSE",
       title="Training-only alpha selection")
ax.legend()
plt.show()

model_results = fit_reward_models(X, Y, train_mask, test_mask, reward_indices, best_alpha)
print(f"Median held-out MSE: {np.median(model_results['mse_full']):.4g}")
print(f"Median held-out R²: {np.nanmedian(model_results['r2_full']):.4g}")

In [ ]:
print(f"Running {N_PERMUTATIONS} trial-wise reward-timing permutations …")
permutation_started = time.perf_counter()
null_delta_mse = reward_permutation_null(
    design, scaler, Y, train_mask, test_mask, beh,
    best_alpha, model_results["mse_reduced"],
    n_permutations=N_PERMUTATIONS, seed=SEED, progress=True,
)
permutation_p = permutation_pvalues(model_results["delta_mse_refit"], null_delta_mse)
reward_candidate = (model_results["delta_mse_refit"] > 0) & (permutation_p < 0.05)
print(f"Permutations completed in {time.perf_counter() - permutation_started:.1f} s")
print(f"Raw p<0.05 candidates: {reward_candidate.sum()} / {len(reward_candidate)}")

## 7. Results and export

We inspect prediction quality, compare the quick zero-out effect with the refit effect,
view example null distributions and reward coefficients, and summarize raw candidate
percentages by region. The exported table preserves every neuron-level metric.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].hist(model_results["mse_full"], bins=40, color="0.25")
axes[0].set(xlabel="held-out MSE", ylabel="neurons", title="Full-model error")

finite_r2 = model_results["r2_full"][np.isfinite(model_results["r2_full"])]
axes[1].hist(finite_r2, bins=40, color="tab:blue")
axes[1].set(xlabel="held-out R²", ylabel="neurons", title="Full-model R²")

# Each dot is one neuron. Above the line, reward timing helped predict it.
sns.stripplot(x=selected_regions, y=model_results["delta_mse_refit"],
              order=list(AREA_CODES), size=3, alpha=0.5, ax=axes[2])
axes[2].axhline(0, color="0.5", linewidth=1)
axes[2].set(xlabel="region", ylabel="ΔMSE refit",
            title="Reward contribution by region")
plt.tight_layout()
plt.show()

In [ ]:
# Ridge.coef_ is (neurons × features) for multi-target Y.
reward_coef = model_results["full_model"].coef_[:, reward_indices]
rank = np.argsort(model_results["delta_mse_refit"])[::-1]
examples = rank[: min(4, len(rank))]

fig, axes = plt.subplots(2, len(examples), figsize=(4 * len(examples), 7))
axes = np.atleast_2d(axes)
lag_centers = np.linspace(-3, 3, reward_coef.shape[1])
for column, neuron in enumerate(examples):
    axes[0, column].hist(null_delta_mse[:, neuron], bins=25, color="0.75")
    axes[0, column].axvline(model_results["delta_mse_refit"][neuron], color="tab:red")
    axes[0, column].set_title(
        f"{selected_regions[neuron]} | p={permutation_p[neuron]:.3f}"
    )
    axes[0, column].set_xlabel("permuted / observed ΔMSE")
    axes[1, column].plot(lag_centers, reward_coef[neuron], marker="o")
    axes[1, column].axhline(0, color="0.5", linewidth=1)
    axes[1, column].set(xlabel="reward lag basis center (s)",
                        ylabel="ridge coefficient")
plt.tight_layout()
plt.show()

In [ ]:
summary_rows = []
for region in AREA_CODES:
    mask = selected_regions == region
    successes, total = int(reward_candidate[mask].sum()), int(mask.sum())
    low, high = proportion_confint(successes, total, method="wilson")
    summary_rows.append({
        "region": region,
        "candidates": successes,
        "neurons": total,
        "percent": 100 * successes / total,
        "ci_low_percent": 100 * low,
        "ci_high_percent": 100 * high,
    })
region_summary = pd.DataFrame(summary_rows)
display(region_summary)

fig, ax = plt.subplots(figsize=(7, 4.5))
centers = region_summary["percent"].to_numpy()
errors = np.vstack([
    centers - region_summary["ci_low_percent"].to_numpy(),
    region_summary["ci_high_percent"].to_numpy() - centers,
])
ax.bar(region_summary["region"], centers,
       color=["#4C78A8", "#72B7B2", "#F2CF5B", "#E45756"])
ax.errorbar(np.arange(len(centers)), centers, yerr=errors, fmt="none",
            color="black", capsize=4)
ax.set(ylabel="raw p<0.05 candidates (%)",
       title="Descriptive reward-encoding candidates by region")
plt.show()
print("WARNING: raw neuron-level p-values are uncorrected; this is one mouse/session.")

In [ ]:
results_df = pd.DataFrame({
    "neuron_index": selected_indices,
    "region": selected_regions,
    "mse_full": model_results["mse_full"],
    "r2_full": model_results["r2_full"],
    "mse_reduced": model_results["mse_reduced"],
    "r2_reduced": model_results["r2_reduced"],
    "delta_mse_refit": model_results["delta_mse_refit"],
    "delta_r2_refit": model_results["delta_r2_refit"],
    "permutation_p": permutation_p,
    "reward_encoding_candidate": reward_candidate,
})
for basis_index in range(reward_coef.shape[1]):
    results_df[f"reward_coef_{basis_index}"] = reward_coef[:, basis_index]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
csv_path = OUTPUT_DIR / "reward_encoding_pilot_results.csv"
json_path = OUTPUT_DIR / "reward_encoding_pilot_run_manifest.json"
results_df.to_csv(csv_path, index=False)

run_seconds = time.perf_counter() - run_started
manifest_out = {
    "session": SESSION,
    "seed": SEED,
    "n_per_region": N_PER_REGION,
    "n_neurons": int(len(selected_indices)),
    "n_permutations": N_PERMUTATIONS,
    "ridge_alpha": best_alpha,
    "train_frames": int(train_mask.sum()),
    "test_frames": int(test_mask.sum()),
    "frame_rate_hz": design.frame_rate,
    "design_columns": int(X.shape[1]),
    "candidate_count": int(reward_candidate.sum()),
    "elapsed_seconds": run_seconds,
    "figshare_files": {
        key: {"file_id": value[0], "filename": value[1], "bytes": value[2]}
        for key, value in FILES.items()
    },
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
    },
}
json_path.write_text(json.dumps(manifest_out, indent=2) + "\n")

print(f"Saved {csv_path} ({len(results_df)} rows)")
print(f"Saved {json_path}")
print(f"Total elapsed time: {run_seconds / 60:.1f} minutes")

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None
if colab_files is not None:
    print("Starting browser downloads for the CSV and run manifest …")
    colab_files.download(str(csv_path))
    colab_files.download(str(json_path))

## 8. Scaling up — reward-encoding neurons across learning (Part B)

Part A above was a deep methods check on **one** session. Here we answer the
proposal's actual question — *how does the proportion of reward-encoding
neurons, and how strongly they activate, change during learning, by region?* —
by applying the **same encoding-model reward ablation** across **many supervised
mice and four learning stages** (`train1` and `train2`, each before vs after
learning).

**What counts as a "reward-encoding" neuron here.** The same definition Part A
validated: fit the full ridge encoding model, then drop the entire
water-delivery reward block and **refit**. `ΔMSE = MSE_reduced − MSE_full` is
reward's unique held-out value for that neuron — the part of its activity that
*only* reward timing can explain, after position, speed, stimulus, licks and the
cue have had their chance to compensate. A neuron is a reward-encoding
**candidate** when `ΔMSE > 0` **and** a permutation test on the exact reward
timing gives `p < 0.05` (reward offsets are shuffled among rewarded trials, so
the null asks specifically about *timing*). Unlike a selectivity index, this
tests that the neuron **specifically encodes reward**, not merely that it
behaves differently on two trial groups.

**Why supervised mice only?** The ablated regressor is *water delivery*, and
unsupervised mice never receive water — their reward block is empty, so their
ablation score would be zero **by construction**, a circular result. The
supervised-vs-unsupervised cohort contrast is exactly what the cue-based
d′ measure was for; here the question is instead *which supervised neurons
specifically encode reward, and how that changes as learning progresses*.

**Two things, tracked across days.** The figure has **two rows**: the
*proportion* of reward-encoding neurons (how *many*, top) and their *mean refit
ΔMSE* (how *strongly* reward contributes, bottom), each plotted across the four
learning stages. **The unit of evidence is the mouse** — every dot is one
mouse's value in one region, and error bars are mean ± s.e.m. **across mice**
(neurons within a mouse are *not* independent replicates — the key limitation
flagged in Part A). The dashed line in the top row is the **nominal 5% chance
floor**: the false-positive rate expected from the per-neuron `p < 0.05`
permutation test alone.

**Runtime budget.** The defaults (2 mice/stage, 250 neurons/region, 50
permutations, fixed ridge alpha) are sized so **Runtime → Run all** finishes in
roughly 20 minutes. Raise `MAX_MICE_PER_CONDITION`, `N_PER_REGION_OVERTIME`, or
`N_PERMUTATIONS_OVERTIME` for tighter error bars and finer p-values.

**Prediction (from the proposal).** Both the proportion *and* the activation of
reward-encoding neurons should *rise* from before→after learning and
concentrate in **anterior** HVAs.

In [ ]:
# --- Part B data access: resolve ANY session's files by name from the deposit ---
# Part A hard-coded three Figshare file ids. Part B visits many sessions, so instead
# we fetch the deposit's file list once and look files up by name.
def fetch_manifest(article_id=ARTICLE_ID):
    """Map every filename in the Figshare deposit to its id and size (one request)."""
    r = requests.get(f"https://api.figshare.com/v2/articles/{article_id}", timeout=(15, 60))
    r.raise_for_status()
    return {f["name"]: {"id": f["id"], "size": f["size"]} for f in r.json()["files"]}


def download_by_name(name, manifest, data_dir=DATA_DIR):
    """Download one deposit file by exact name; cached and size-verified."""
    data_dir = Path(data_dir); data_dir.mkdir(parents=True, exist_ok=True)
    if name not in manifest:
        raise KeyError(f"{name} is not in the Figshare deposit")
    target = data_dir / name
    expected = manifest[name]["size"]
    if target.exists() and target.stat().st_size == expected:
        return target
    url = f"https://ndownloader.figshare.com/files/{manifest[name]['id']}"
    with requests.get(url, stream=True, timeout=(15, 600)) as r:
        r.raise_for_status()
        with target.open("wb") as fh:
            for chunk in r.iter_content(1024 * 1024):
                fh.write(chunk)
    if target.stat().st_size != expected:
        raise RuntimeError(f"Truncated download for {name}; rerun the cell to retry.")
    return target

In [ ]:
def reward_ablation_session(beh, mouse, manifest,
                            n_per_region=N_PER_REGION_OVERTIME,
                            n_permutations=N_PERMUTATIONS_OVERTIME,
                            alpha=PART_B_ALPHA, seed=SEED):
    """Reward encoding per sampled neuron for one supervised mouse, by region.

    Part A's encoding-model ablation applied to one session: build the expanded
    design matrix, fit the full ridge model and a model with the reward block
    removed (alternating train/test trials), then permutation-test the exact
    reward timing. A neuron is a reward-encoding **candidate** when
    `delta_mse_refit > 0` and the permutation `p < 0.05`; its **strength** (the
    "activation" tracked across days) is `delta_mse_refit` itself, reported for
    *all* sampled neurons so it is not biased by the selection. The ablated
    block is WATER delivery, so this is only defined for supervised mice.

    Returns per-region candidate-flag arrays and strength arrays.
    """
    # --- neurons: sample per region, reconstruct ONLY those from the SVD (RAM-safe) ---
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Trying to unpickle estimator")
        svd = np.load(download_by_name(f"{mouse}_SVD_dec.npy", manifest),
                      allow_pickle=True).item()
    retin = np.load(download_by_name(f"{mouse[:-1]}trans.npz", manifest), allow_pickle=False)
    sel, sel_regions = balanced_sample(map_regions(retin["iarea"]), n_per_region, seed=seed)
    Y = reconstruct_selected(svd, sel)                        # frames × neurons
    del svd
    if not np.isfinite(Y).all():
        raise ValueError("reconstructed neural targets contain non-finite values")

    # --- behaviour + design matrix (the same pipeline as Part A) ---
    beh = align_behavior_frames(beh, Y.shape[0])
    if np.isfinite(beh["RewardFr"]).sum() < 10:
        raise ValueError("too few reward events for a reward-timing permutation test")
    train_mask, test_mask = odd_even_masks(beh["ft_trInd"])
    design = build_design_matrix(beh, train_mask)
    scaler = StandardScaler().fit(design.raw[train_mask])     # training frames only
    X = scaler.transform(design.raw)
    reward_indices = design.blocks["reward"]

    # --- full vs reward-ablated refits + reward-timing permutation null ---
    results = fit_reward_models(X, Y, train_mask, test_mask, reward_indices, alpha)
    null = reward_permutation_null(
        design, scaler, Y, train_mask, test_mask, beh,
        alpha, results["mse_reduced"],
        n_permutations=n_permutations, seed=seed, progress=False,
    )
    p = permutation_pvalues(results["delta_mse_refit"], null)
    candidate = (results["delta_mse_refit"] > 0) & (p < 0.05)
    return {
        "flag_by_region": {r: candidate[sel_regions == r] for r in AREA_CODES},
        "strength_by_region": {r: results["delta_mse_refit"][sel_regions == r]
                               for r in AREA_CODES},
    }

In [ ]:
# --- Run the reward-ablation analysis across stages and mice (supervised only) ---
# Downloads one SVD + retinotopy file per mouse (~100-180 MB each, cached), then
# refits the encoding model with and without the reward block per session. This is
# the slow cell; the defaults (2 mice/stage, 250 neurons/region, 50 permutations)
# are sized for a ~20-minute Run-all — raise them in the config for more power.
manifest = fetch_manifest()
rng = np.random.default_rng(SEED)
overtime_rows, skipped = [], []

for cohort, stage, beh_file in OVERTIME_CONDITIONS:
    beh_all = np.load(download_by_name(beh_file, manifest), allow_pickle=True).item()
    # a mouse is usable only if its SVD and retinotopy files are also in the deposit
    mice = [m for m in beh_all
            if f"{m}_SVD_dec.npy" in manifest and f"{m[:-1]}trans.npz" in manifest]
    mice = mice[:MAX_MICE_PER_CONDITION]
    print(f"\n{cohort} / {stage}: {len(mice)} mice")
    for mouse in mice:
        try:
            res = reward_ablation_session(beh_all[mouse], mouse, manifest)
        except (KeyError, ValueError, RuntimeError) as err:
            skipped.append((cohort, stage, mouse, str(err)))
            print(f"  skip {mouse}: {err}")
            continue
        for region in AREA_CODES:                       # one row per (mouse, region)
            flag = res["flag_by_region"][region]
            strength = res["strength_by_region"][region]
            if len(flag):
                overtime_rows.append({
                    "cohort": cohort, "stage": stage, "region": region, "mouse": mouse,
                    "pct_reward": 100 * np.mean(flag),            # how MANY encode reward
                    "mean_delta_mse": float(np.mean(strength)),   # how STRONGLY (activation)
                    "n_neurons": len(flag),
                })
        print(f"  {mouse}: ok")

overtime_df = pd.DataFrame(overtime_rows)
CHANCE_PCT = 5.0  # nominal false-positive rate of the per-neuron p<0.05 permutation test
n_mice = overtime_df["mouse"].nunique() if len(overtime_df) else 0
print(f"\n{len(overtime_df)} region×mouse points from {n_mice} mice; "
      f"nominal chance floor = {CHANCE_PCT:.0f}% (permutation p<0.05)")
if skipped:
    print(f"{len(skipped)} session(s) skipped — inspect the `skipped` list for reasons.")
if len(overtime_df):
    display(overtime_df.groupby(["stage", "region"])[["pct_reward", "mean_delta_mse"]]
            .mean().round(3))

In [ ]:
# --- Figure: reward-encoding across learning, by region (each dot = one mouse) ---
# Top row  = how MANY neurons encode reward (proportion of ablation candidates).
# Bottom row = how STRONGLY (mean refit ΔMSE = activation). Lines join the
# per-stage means so you can read the learning trajectory. Supervised cohort only.
if len(overtime_df) == 0:
    print("No sessions produced results — raise MAX_MICE_PER_CONDITION or check `skipped`.")
else:
    stages = [s for s in STAGE_ORDER if s in set(overtime_df["stage"])]
    xpos = {s: i for i, s in enumerate(stages)}
    metrics = [("pct_reward", "% reward-encoding neurons\n(ablation, p < 0.05)", True),
               ("mean_delta_mse", "mean refit ΔMSE\n(activation strength)", False)]
    fig, axes = plt.subplots(2, 4, figsize=(16, 7.5), sharex=True)
    for row, (key, ylab, show_floor) in enumerate(metrics):
        for col, region in enumerate(AREA_CODES):
            ax = axes[row, col]
            means, xs = [], []
            for s in stages:
                pts = overtime_df[(overtime_df.region == region) &
                                  (overtime_df.stage == s)][key].to_numpy()
                if not len(pts):
                    continue
                x = xpos[s]
                ax.scatter(x + rng.uniform(-0.05, 0.05, len(pts)), pts,   # jittered dots
                           color="#4C72B0", alpha=0.6, s=30, edgecolor="none", zorder=3)
                sem = pts.std(ddof=1) / np.sqrt(len(pts)) if len(pts) > 1 else 0.0
                ax.errorbar(x, pts.mean(), yerr=sem, fmt="_", color="#4C72B0",
                            markersize=20, lw=2, capsize=4, zorder=4)
                means.append(pts.mean()); xs.append(x)
            if len(xs) > 1:
                ax.plot(xs, means, color="#4C72B0", lw=1.2, alpha=0.8, zorder=2)  # trajectory
            if show_floor:
                ax.axhline(CHANCE_PCT, ls="--", color="0.5", lw=1)
            if row == 0:
                ax.set_title(region, fontweight="bold")
            ax.set_xticks(range(len(stages)))
            ax.set_xticklabels([s.replace("_", "\n") for s in stages], fontsize=8)
            ax.set_xlim(-0.5, len(stages) - 0.5)
            ax.spines[["top", "right"]].set_visible(False)
        axes[row, 0].set_ylabel(ylab)
    handles = [plt.Line2D([], [], marker="o", ls="-", color="#4C72B0", label="supervised"),
               plt.Line2D([], [], ls="--", color="0.5", label="chance (5% nominal)")]
    axes[0, -1].legend(handles=handles, fontsize=8, loc="upper left", frameon=False)
    fig.suptitle("Reward-encoding neurons across learning — proportion (top) and "
                 "activation (bottom); each dot = one mouse (supervised, reward ablation)",
                 y=1.01)
    plt.tight_layout(); plt.show()

    csv_path = OUTPUT_DIR / "reward_encoding_over_time.csv"
    overtime_df.to_csv(csv_path, index=False)
    print(f"Saved {csv_path} ({len(overtime_df)} rows)")
    try:
        from google.colab import files as colab_files
        colab_files.download(str(csv_path))
    except ImportError:
        pass

## Interpretation checklist

**Part A (one-session ablation).**
- Positive refit ΔMSE means the reward block improved held-out prediction beyond
  the other regressors; negative values are allowed and should not be forced to zero.
- A raw permutation `p < 0.05` candidate is exploratory. With ~3,000 tests, a few
  dozen chance positives are expected under a complete null.
- Neurons are not independent replicates, so Part A's region percentages are
  descriptive only — this is exactly what Part B fixes.
- SVD-reconstructed deconvolved activity is the target; it is not raw ΔF/F.

**Part B (reward ablation across learning, supervised mice).**
- Two questions, two rows: **how many** neurons encode reward (proportion of
  ablation candidates, top) and **how strongly** reward contributes (mean refit
  ΔMSE, bottom), across the four `train1`/`train2` before/after stages.
- Read the top row against the **dashed 5% line** — the nominal false-positive
  rate of the per-neuron `p < 0.05` permutation test. A region/stage is only
  interesting if its mean sits clearly above it; the signal is the per-mouse
  *proportion* above that floor, not any single neuron.
- The **mouse** is the replicate: error bars are s.e.m. across mice, and with
  `MAX_MICE_PER_CONDITION = 2` they will be very wide — resist over-reading a
  single step of the trajectory; raise the cap for tighter bars (at the cost of
  runtime).
- **Supervised cohort only**, by construction: the ablated regressor is water
  delivery, which unsupervised mice never receive, so their ablation would score
  zero trivially. This notebook therefore does *not* test the cohort contrast —
  it tracks reward encoding across learning within the supervised cohort.
- Candidates carry no multiple-comparison correction across neurons, and a fixed
  ridge alpha (`PART_B_ALPHA`) is used for speed — treat borderline percentages
  near the 5% floor as noise.
- Knobs: `MAX_MICE_PER_CONDITION`, `N_PER_REGION_OVERTIME`, and
  `N_PERMUTATIONS_OVERTIME` trade statistical power for the ~20-minute Run-all
  budget.